# 2. Can I tag BIB with five jet variables?

I start with whole jets. My signal is simulated jets matched to long-lived particles (LLPs). For beam-induced background (BIB), I use jets from unpaired bunch crossings. Other backgrounds can end up there too.

## How does XGBoost pick a split?

Take four example jets. Two LLP jets have times of 0 and 1 ns, and two unpaired jets have times of 5 and 7 ns. Start each BIB score at $s=0.5$. The labels are $y=0$ for LLP and $y=1$ for unpaired.

For each split, add up $g_i=s_i-y_i$ and $h_i=s_i(1-s_i)$ on each side. Call the sums $G$ and $H$. With unit weights, $\lambda=1$ and no split penalty, the gain is

$$
\frac12\left(\frac{G_L^2}{H_L+1}+\frac{G_R^2}{H_R+1}
-\frac{G_P^2}{H_P+1}\right).
$$

$P$ is the unsplit group. At 3 ns, $G_L=1$, $G_R=-1$ and $H_L=H_R=0.5$. The gain is $2/3$, versus $0.17$ at 0.5 ns. Of these two cuts, 3 ns wins. Any cut between 1 and 5 ns gives the same groups. Each leaf adds $-G/(H+1)$ times the learning rate to the log-odds. The LLP scores drop and the unpaired scores rise. The next tree works from those updated scores. [XGBoost derivation](https://xgboost.readthedocs.io/en/stable/tutorials/model.html)

## Which jets go in?

I keep jets with a valid ATLAS time, $p_T\geq30$ GeV, $|\eta|<1.7$ and a Tile energy fraction between 0 and 1, excluding both ends. LLP jets must pass the stored truth match. Here I use unpaired events with any vertex count.

The five inputs are jet time, time spread, time uncertainty (all in ns), cluster count and Tile energy fraction. A second model sees just transverse momentum $p_T$ and direction $|\eta|$. This checks how much those two variables reveal on their own. [Cells and selections](01_cells_and_selections.ipynb)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json

import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import uproot
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping

here = Path.cwd()
project = next(p for p in (here, *here.parents) if (p / "notebooks").is_dir())
data_folder = project / "DataForEmre"
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
run_dir = project / "analysis/outputs/02_xgboost_baseline" / stamp
run_dir.mkdir(parents=True, exist_ok=False)

In [ ]:
physics_features = [
    "jet_time_ATLAS", "jet_time_spread", "jet_time_sigma",
    "jet_n_clusters", "jet_TileEnergyFrac",
]
jet_fields = ["jet_pt", "jet_eta", *physics_features]
event_fields = ["runNumber", "eventNumber"]

source_folders = {
    "LLP": data_folder / "collision/LLP/N1N2_rpv_1400_10ns",
    "unpaired": data_folder / "non_collision/unpaired/physics_Background_NTuple_unsplit_v2",
}

In [ ]:
def jet_table(batch, keys, first_row):
    counts = ak.to_numpy(ak.num(batch["jet_pt"], axis=1))
    columns = {}
    for field in jet_fields:
        np.testing.assert_array_equal(ak.to_numpy(ak.num(batch[field], axis=1)), counts)
        values = ak.to_numpy(ak.flatten(batch[field], axis=1))
        columns[field] = np.ma.asarray(values, dtype=float).filled(np.nan)

    table = pd.DataFrame(columns).replace([-999999.0, np.inf, -np.inf], np.nan)
    table["event_group"] = np.repeat(keys, counts)
    table["row_index"] = np.repeat(np.arange(first_row, first_row + len(batch)), counts)
    table["jet_index"] = ak.to_numpy(ak.flatten(ak.local_index(batch["jet_pt"], axis=1)))
    table["jet_pt_GeV"] = table["jet_pt"] / 1000
    table["jet_abs_eta"] = table["jet_eta"].abs()
    return table

In [ ]:
def mark_missing_inputs(table):
    # XGBoost can take missing values for these three inputs.
    table.loc[table["jet_time_spread"] < 0, "jet_time_spread"] = np.nan
    table.loc[table["jet_time_sigma"] <= 0, "jet_time_sigma"] = np.nan
    clusters = table["jet_n_clusters"]
    table.loc[(clusters < 1) | (clusters % 1 != 0), "jet_n_clusters"] = np.nan

In [ ]:
def select_batch(table, batch, sample):
    truth = np.ones(len(table), dtype=bool)
    if sample == "LLP":
        truth = ak.to_numpy(ak.flatten(ak.fill_none(batch["jet_is_llp_matched"], 0))) == 1
    tile = table["jet_TileEnergyFrac"]
    cuts = {
        "LLP match": truth,
        "Valid pT, eta and time": table[["jet_pt_GeV", "jet_eta", "jet_time_ATLAS"]].notna().all(axis=1),
        "pT >= 30 GeV": table["jet_pt_GeV"].ge(30),
        "abs(eta) < 1.7": table["jet_abs_eta"].lt(1.7),
        "Energy in both calorimeters": tile.gt(0) & tile.lt(1),
    }
    keep = np.ones(len(table), dtype=bool)
    flow = {"All jets": len(table)}
    for name, condition in cuts.items():
        keep &= np.asarray(condition, dtype=bool)
        flow[name] = int(keep.sum())
    return table.loc[keep], flow

In [ ]:
def read_source(path, sample, seen_events):
    fields = [*event_fields, *jet_fields]
    if sample == "LLP":
        fields.append("jet_is_llp_matched")
    first_row = 0
    with uproot.open(path) as source:
        tree = source["JetContent"]
        for batch in tree.iterate(fields, step_size=10_000, library="ak"):
            keys = [f"{sample}:{run}:{event}" for run, event in
                    zip(batch["runNumber"], batch["eventNumber"])]
            assert len(set(keys)) == len(keys) and seen_events.isdisjoint(keys), path
            seen_events.update(keys)
            table = jet_table(batch, keys, first_row)
            mark_missing_inputs(table)
            selected, counts = select_batch(table, batch, sample)
            selected = selected.assign(sample=sample, label=int(sample == "unpaired"),
                                       source_file=str(path.relative_to(data_folder)))
            yield selected, counts
            first_row += len(batch)

In [ ]:
parts, cut_rows = [], []
for sample, folder in source_folders.items():
    files = sorted(folder.glob("*.root"))
    assert files, f"No ROOT files in {folder}"
    seen_events = set()
    for path in files:
        for selected, counts in read_source(path, sample, seen_events):
            parts.append(selected)
            cut_rows.append({"sample": sample, **counts})

jets = pd.concat(parts, ignore_index=True)
assert not jets.duplicated(["event_group", "jet_index"]).any()
cutflow = pd.DataFrame(cut_rows).groupby("sample").sum().T
display(cutflow)
cutflow.to_csv(run_dir / "cutflow.csv")

## How I train the trees

I split by event so jets from one event stay together. I use 60% to train, 20% for validation and 20% for the test. A training jet gets weight $N/(2N_c)$, where $N_c$ is its class count. Both classes then have equal total weight.

High scores are BIB-like. I sort all validation LLP scores and put the cut just above the lowest-scoring 95%. That keeps at least 95% by jet count, including ties. I keep the same cut on test.

In [ ]:
seed = 42
events = jets[["event_group", "label"]].drop_duplicates().sort_values("event_group")
train_val, test_events = train_test_split(
    events, test_size=0.20, random_state=seed, stratify=events["label"],
)
train_events, val_events = train_test_split(
    train_val, test_size=0.25, random_state=seed, stratify=train_val["label"],
)
split_frames = {"train": train_events, "validation": val_events, "test": test_events}
assignment = {group: split for split, frame in split_frames.items()
              for group in frame["event_group"]}
jets["split"] = jets["event_group"].map(assignment)
assert jets["split"].notna().all()
assert jets.groupby("event_group")["split"].nunique().eq(1).all()
table = jets.groupby(["split", "sample"]).agg(jets=("label", "size"), events=("event_group", "nunique"))
display(table)

In [ ]:
model_features = {
    "XGBoost": physics_features,
    "Kinematics only": ["jet_pt_GeV", "jet_abs_eta"],
}
parameters = dict(
    n_estimators=600, learning_rate=0.05, max_depth=3, min_child_weight=10,
    subsample=0.8, reg_lambda=5.0, objective="binary:logistic", eval_metric="logloss",
    tree_method="hist", n_jobs=4, random_state=seed, base_score=0.5,
)

def fit_trees(train, validation, features):
    class_weight = len(train) / (2 * train["label"].value_counts())
    train_weight = train["label"].map(class_weight)
    val_weight = validation["label"].map(class_weight)
    stop = EarlyStopping(rounds=40, data_name="validation_1", metric_name="logloss", save_best=True)
    model = XGBClassifier(**parameters, callbacks=[stop])
    model.fit(
        train[features], train["label"], sample_weight=train_weight,
        eval_set=[(train[features], train["label"]), (validation[features], validation["label"])],
        sample_weight_eval_set=[train_weight, val_weight], verbose=False,
    )
    return model

In [ ]:
def validation_cut(model, validation, features):
    signal = validation.loc[validation["label"] == 0, features]
    scores = np.sort(model.predict_proba(signal)[:, 1].astype(np.float64))
    keep_count = int(np.ceil(0.95 * len(scores)))
    return float(np.nextafter(scores[keep_count - 1], np.inf))

## What happens on test?

I count unpaired jets tagged and LLP jets kept, first across all times and then for delayed jets ($t_{\rm ATLAS}>3$ ns). AUC measures how often a random unpaired jet outscores a random LLP jet, with half credit for ties.

In [ ]:
def score_metrics(labels, scores, cut):
    labels = np.asarray(labels)
    unpaired = labels == 1
    tagged = scores >= cut
    rejected = int((tagged & unpaired).sum())
    retained = int((~tagged & ~unpaired).sum())
    return {
        "Unpaired tagged": rejected, "Unpaired total": int(unpaired.sum()),
        "LLP kept": retained, "LLP total": int((~unpaired).sum()),
        "ROC AUC": roc_auc_score(labels, scores) if np.unique(labels).size == 2 else np.nan,
    }

def test_rows(frame, scores, cut, name, selection):
    regions = {"All times": np.ones(len(frame), dtype=bool),
               "ATLAS time > 3 ns": frame["jet_time_ATLAS"].gt(3).to_numpy()}
    return [{"Model": name, "Sample": selection, "Region": region,
             **score_metrics(frame.loc[mask, "label"], scores[mask], cut)}
            for region, mask in regions.items()]

In [ ]:
def run_comparison(data, selection):
    train = data.loc[data["split"] == "train"]
    validation = data.loc[data["split"] == "validation"]
    test = data.loc[data["split"] == "test"]
    predictions = test[["source_file", "row_index", "jet_index", "event_group", "label"]].copy()

    models, cuts, rows = {}, {}, []
    for name, features in model_features.items():
        model = fit_trees(train, validation, features)
        cut = validation_cut(model, validation, features)
        scores = model.predict_proba(test[features])[:, 1].astype(np.float64)
        predictions[name] = scores
        models[name], cuts[name] = model, cut
        rows.extend(test_rows(test, scores, cut, name, selection))
        print(f"{name}: {model.best_iteration + 1} rounds, cut = {cut:.5f}")
    return models, cuts, predictions, pd.DataFrame(rows)

models, cuts, predictions, broad_results = run_comparison(jets, "Broad")
display(broad_results.round(4))

## What if both samples have similar jets?

Even $p_T$ and $|\eta|$ alone give an AUC of 0.99 on the full test sample. I repeat the comparison using bins of $p_T$ and $|\eta|$ with at least three jets of each class in training. Within each bin I draw equal numbers from both classes, separately for train, validation and test. Then I refit both models and their cuts.

In [ ]:
pt_edges = np.geomspace(30, 3000, 22)
eta_edges = np.linspace(0, 1.7, 10)
binned = jets.assign(pt_bin=pd.cut(jets["jet_pt_GeV"], pt_edges, labels=False, include_lowest=True),
                     eta_bin=pd.cut(jets["jet_abs_eta"], eta_edges, labels=False, include_lowest=True))
bin_columns = ["pt_bin", "eta_bin"]
train_counts = (binned.loc[binned["split"].eq("train")]
                .groupby([*bin_columns, "label"]).size().unstack(fill_value=0))
supported = train_counts.index[train_counts.min(axis=1).ge(3)]
in_support = pd.MultiIndex.from_frame(binned[bin_columns]).isin(supported)
matched_parts = []
for _, group in binned.loc[in_support].groupby(["split", *bin_columns]):
    count = group["label"].value_counts().reindex([0, 1], fill_value=0).min()
    if count:
        matched_parts.append(group.groupby("label").sample(n=count, random_state=seed))
matched = pd.concat(matched_parts).sort_index()
balance = matched.groupby(["split", *bin_columns, "label"]).size().unstack(fill_value=0)
assert balance[0].eq(balance[1]).all()
display(matched.groupby(["split", "sample"]).size().unstack())

In [ ]:
matched_models, matched_cuts, matched_predictions, matched_results = run_comparison(matched, "Matched")
display(matched_results.round(4))
summary = pd.concat([broad_results, matched_results], ignore_index=True)

## When does more training stop helping?

For the matched jets, training loss keeps falling after about 100 rounds, while validation loss stays near 0.40. The extra trees mostly help on jets the model has seen. The dashed line marks the round with the lowest validation loss.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for axis, model, title in zip(axes, [models["XGBoost"], matched_models["XGBoost"]],
                              ["Broad selection", "Matched kinematics"]):
    curves = model.evals_result()
    for key, split in [("validation_0", "Train"), ("validation_1", "Validation")]:
        loss = curves[key]["logloss"]
        axis.plot(np.arange(1, len(loss) + 1), loss, label=split)
    axis.axvline(model.best_iteration + 1, color="grey", linestyle="--", label="Kept round")
    axis.set(title=title, xlabel="Boosting round", ylabel="Weighted log loss")
    axis.legend()
fig.tight_layout()
fig.savefig(run_dir / "learning_curves.png", dpi=150)
plt.show()
plt.close(fig)

In [ ]:
for label, study_models in [("broad", models), ("matched", matched_models)]:
    for name, model in study_models.items():
        filename = label + "_" + name.lower().replace(" ", "_") + ".ubj"
        model.save_model(run_dir / filename)
settings = {"seed": seed, "features": model_features, "parameters": parameters,
            "broad_cuts": cuts, "matched_cuts": matched_cuts,
            "pt_edges_GeV": pt_edges.tolist(), "abs_eta_edges": eta_edges.tolist(),
            "supported_bins": supported.to_frame(index=False).to_dict(orient="records")}
(run_dir / "settings.json").write_text(json.dumps(settings, indent=2))
summary.to_csv(run_dir / "test_summary.csv", index=False)
jets[["event_group", "label", "split"]].drop_duplicates().to_csv(run_dir / "event_split.csv.gz", index=False)
predictions.to_csv(run_dir / "broad_scores.csv.gz", index=False)
matched_predictions.to_csv(run_dir / "matched_scores.csv.gz", index=False)
print("New results:", run_dir)

After matching, the kinematics-only AUC drops to **0.59**. The five jet variables give **0.90** across all times. But at the validation cut they tag **0 of 18** delayed unpaired jets and keep **104 of 104** delayed LLP jets. That small delayed sample is why I look at what individual cell times and positions can add. [Train and compare](03_train_and_compare.ipynb)